# EDA — Vietnamese Financial News & Market Pipeline

Notebook kiểm tra chất lượng Silver/Gold, ticker extraction, market data, point-in-time join và event window. Các kết quả hiện tại chỉ mang tính mô tả, không chứng minh quan hệ nhân quả.

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 100)
ROOT = Path.cwd()
NEWS_PATH = ROOT / 'data_lake/silver/news'
MARKET_PATH = ROOT / 'data_lake/silver/market_daily'
GOLD_PATH = ROOT / 'data_lake/gold/news_market_impact'
for path in (NEWS_PATH, MARKET_PATH, GOLD_PATH):
    if not path.exists():
        raise FileNotFoundError(f'Missing dataset: {path}')
print('Project root:', ROOT)

In [ ]:
news = pd.read_parquet(NEWS_PATH)
market = pd.read_parquet(MARKET_PATH)
gold = pd.read_parquet(GOLD_PATH)
for col in ['published_timestamp', 'collected_timestamp']:
    news[col] = pd.to_datetime(news[col], utc=True, errors='coerce')
market['session_date'] = pd.to_datetime(market['session_date'].astype(str), errors='coerce')
gold['event_at_vn'] = pd.to_datetime(gold['event_at_vn'], errors='coerce')
for frame in (news, market, gold):
    for col in frame.select_dtypes(include=['object']).columns:
        if frame[col].map(lambda x: isinstance(x, np.ndarray)).any():
            frame[col] = frame[col].map(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
print(f'News={len(news):,} | Market={len(market):,} | Gold={len(gold):,}')

## 1. Data quality và coverage tổng quan

In [ ]:
news_quality = pd.Series({
    'rows': len(news),
    'distinct_event_id': news['event_id'].nunique(),
    'duplicate_event_id': news.duplicated('event_id').sum(),
    'duplicate_url': news['url'].dropna().duplicated().sum(),
    'duplicate_content_hash': news['content_hash'].dropna().duplicated().sum(),
    'invalid_rows': (~news['is_valid']).sum(),
    'missing_title': news['title'].fillna('').str.strip().eq('').sum(),
    'has_full_text': news['has_full_text'].sum(),
    'has_image': news['image_url'].fillna('').str.strip().ne('').sum(),
    'has_author': news['author'].fillna('').str.strip().ne('').sum(),
    'has_vn30_ticker': news['merged_tickers'].map(len).gt(0).sum(),
}, name='value').to_frame()
display(news_quality)
source_summary = news.groupby('source').agg(
    articles=('event_id', 'size'),
    full_text=('has_full_text', 'sum'),
    ticker_articles=('merged_tickers', lambda s: s.map(len).gt(0).sum()),
).sort_values('articles', ascending=False)
source_summary['full_text_rate_pct'] = (100 * source_summary.full_text / source_summary.articles).round(1)
source_summary['ticker_coverage_pct'] = (100 * source_summary.ticker_articles / source_summary.articles).round(1)
display(source_summary)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
source_summary['articles'].sort_values().plot.barh(ax=axes[0], title='Số bài theo nguồn')
news['content_length'].plot.hist(bins=30, ax=axes[1], title='Phân bố độ dài content')
news.assign(published_date=news.published_timestamp.dt.tz_convert('Asia/Ho_Chi_Minh').dt.date) \
    .groupby('published_date').size().plot(ax=axes[2], marker='o', title='Số bài theo ngày')
plt.tight_layout(); plt.show()
latency = (news['collected_timestamp'] - news['published_timestamp']).dt.total_seconds() / 60
latency_valid = latency[(latency >= 0) & (latency < 7 * 24 * 60)]
display(pd.Series({
    'latency_valid_rows': latency_valid.size,
    'median_minutes': latency_valid.median(),
    'p95_minutes': latency_valid.quantile(.95),
    'negative_latency_rows': (latency < 0).sum(),
}, name='publish_to_collect').round(2).to_frame())

## 2. Ticker extraction: rule-based vs CRF NER

In [ ]:
ticker_eval = news[['event_id', 'source', 'rulebased_tickers', 'ner_tickers', 'merged_tickers']].copy()
ticker_eval['rule_set'] = ticker_eval.rulebased_tickers.map(set)
ticker_eval['ner_set'] = ticker_eval.ner_tickers.map(set)
ticker_eval['group'] = np.select([
    ticker_eval.rule_set.map(bool) & ticker_eval.ner_set.map(bool) & (ticker_eval.rule_set == ticker_eval.ner_set),
    ticker_eval.rule_set.map(bool) & ~ticker_eval.ner_set.map(bool),
    ~ticker_eval.rule_set.map(bool) & ticker_eval.ner_set.map(bool),
    ticker_eval.rule_set.map(bool) & ticker_eval.ner_set.map(bool),
], ['rule_ner_agree', 'rule_only', 'ner_only', 'partial_disagreement'], default='no_ticker')
display(ticker_eval.group.value_counts().rename('articles').to_frame())
ticker_counts = (news[['event_id', 'merged_tickers']].explode('merged_tickers')
                 .dropna().groupby('merged_tickers').event_id.nunique().sort_values(ascending=False))
ax = ticker_counts.head(20).sort_values().plot.barh(figsize=(9, 6), title='Top ticker theo số bài')
ax.set_xlabel('Số bài'); plt.tight_layout(); plt.show()
review = ticker_eval[ticker_eval.group.isin(['ner_only', 'partial_disagreement'])][
    ['event_id', 'source', 'rulebased_tickers', 'ner_tickers', 'merged_tickers', 'group']
]
display(review.head(30))

## 3. Market Silver: coverage, return và correlation

In [ ]:
market_quality = pd.Series({
    'rows': len(market),
    'tickers': market.ticker.nunique(),
    'sessions': market.session_date.nunique(),
    'duplicate_keys': market.duplicated(['ticker','session_date','interval','source']).sum(),
    'invalid_rows': (~market.is_valid).sum(),
    'weekend_rows': market.session_date.dt.dayofweek.ge(5).sum(),
    'missing_return': market.daily_return.isna().sum(),
    'abs_return_gt_10pct': market.daily_return.abs().gt(.10).sum(),
}, name='value').to_frame()
display(market_quality)
coverage = market.groupby('ticker').agg(
    sessions=('session_date','nunique'),
    first_session=('session_date','min'),
    last_session=('session_date','max'),
    volatility=('daily_return','std'),
    mean_volume=('volume','mean'),
).sort_values('volatility', ascending=False)
display(coverage.head(15))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(market.daily_return.dropna() * 100, bins=40, kde=True, ax=axes[0])
axes[0].set(title='Daily return VN30', xlabel='Return (%)')
normalized = market.pivot(index='session_date', columns='ticker', values='close')
normalized = 100 * normalized / normalized.iloc[0]
normalized.plot(ax=axes[1], legend=False, alpha=.6, title='Normalized close (base=100)')
plt.tight_layout(); plt.show()
returns_wide = market.pivot(index='session_date', columns='ticker', values='daily_return')
plt.figure(figsize=(12, 9))
sns.heatmap(returns_wide.corr(), cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Correlation daily return'); plt.tight_layout(); plt.show()

## 4. Audit timezone và point-in-time join

In [ ]:
audit_cols = ['event_id','ticker','news_source','event_at_vn','event_date_vn',
              'published_after_close','anchor_date_t_minus_1','t0_date']
audit = gold[audit_cols].drop_duplicates(['event_id','ticker']).copy()
audit['event_date_vn'] = pd.to_datetime(audit.event_date_vn.astype(str))
audit['t0_date'] = pd.to_datetime(audit.t0_date.astype(str))
audit['anchor_date_t_minus_1'] = pd.to_datetime(audit.anchor_date_t_minus_1.astype(str))
audit['bad_t0'] = np.where(
    audit.published_after_close, audit.t0_date <= audit.event_date_vn,
    audit.t0_date < audit.event_date_vn
)
audit['bad_anchor'] = audit.anchor_date_t_minus_1 >= audit.t0_date
display(pd.Series({
    'event_ticker_pairs': len(audit),
    'published_after_close': audit.published_after_close.sum(),
    'bad_t0': audit.bad_t0.sum(),
    'bad_anchor': audit.bad_anchor.sum(),
}, name='audit').to_frame())
display(audit.sort_values('event_at_vn').head(20))

## 5. Event-window EDA: availability, AAR và abnormal return

In [ ]:
coverage_by_t = gold.groupby('relative_session').agg(
    requested=('event_id','size'),
    available=('has_market_data','sum'),
).sort_index()
coverage_by_t['availability_pct'] = (100 * coverage_by_t.available / coverage_by_t.requested).round(1)
display(coverage_by_t)
observed = gold[gold.has_market_data & gold.abnormal_return.notna()].copy()
event_stats = observed.groupby('relative_session').abnormal_return.agg(['count','mean','median','std'])
event_stats['se'] = event_stats['std'] / np.sqrt(event_stats['count'])
event_stats['ci95_low'] = event_stats['mean'] - 1.96 * event_stats['se']
event_stats['ci95_high'] = event_stats['mean'] + 1.96 * event_stats['se']
display((event_stats * pd.Series({'count':1,'mean':100,'median':100,'std':100,'se':100,'ci95_low':100,'ci95_high':100})).round(3))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
coverage_by_t.available.plot.bar(ax=axes[0], title='Số event có dữ liệu theo T')
x = event_stats.index.to_numpy()
axes[1].plot(x, event_stats['mean'] * 100, marker='o', label='Mean abnormal return')
axes[1].plot(x, event_stats['median'] * 100, marker='s', label='Median')
axes[1].fill_between(x, event_stats.ci95_low * 100, event_stats.ci95_high * 100, alpha=.2, label='95% normal CI')
axes[1].axhline(0, color='black', lw=1); axes[1].set(xlabel='Relative session', ylabel='%')
axes[1].legend(); axes[1].set_title('Abnormal return từ close T-1')
plt.tight_layout(); plt.show()

In [ ]:
t1 = observed[observed.relative_session.isin([0,1,3,5])].copy()
source_effect = t1.groupby(['news_source','relative_session']).agg(
    events=('event_id','nunique'), mean_abnormal=('abnormal_return','mean'),
    median_abnormal=('abnormal_return','median')
).reset_index()
source_effect[['mean_abnormal','median_abnormal']] *= 100
display(source_effect.round(3))
timing_effect = t1.groupby(['published_after_close','relative_session']).agg(
    events=('event_id','nunique'), mean_abnormal=('abnormal_return','mean')
).reset_index()
timing_effect['mean_abnormal'] *= 100
display(timing_effect.round(3))

## 6. Storage inventory và kết luận readiness

In [ ]:
def inventory(path):
    files = list(path.rglob('*.parquet'))
    return {'parquet_files': len(files), 'size_kib': round(sum(f.stat().st_size for f in files)/1024, 1)}
storage = pd.DataFrame({
    'silver_news': inventory(NEWS_PATH),
    'silver_market': inventory(MARKET_PATH),
    'gold_impact': inventory(GOLD_PATH),
}).T
display(storage)
readiness = pd.Series({
    'news_rows': len(news),
    'ticker_articles': news.merged_tickers.map(len).gt(0).sum(),
    'event_ticker_pairs': gold[['event_id','ticker']].drop_duplicates().shape[0],
    'complete_T_plus_5': gold.query('relative_session == 5 and has_market_data').shape[0],
    'market_sessions': market.session_date.nunique(),
}, name='current_dataset').to_frame()
display(readiness)
print('Kết luận: đủ để audit pipeline và EDA mô tả; chưa đủ để train mô hình dự đoán đáng tin cậy.')